In [ ]:
def trigger_working_set_manager(state):
    print(f"ALERT!: Thrashing detected with severity {state:.3f}! Triggering memory control module Immediately")
import time
import joblib
import psutil
import pandas as pd
from collections import deque
model = joblib.load("thrashing_detection_model.pkl")
scaler = joblib.load("thrashing_detection_scaler.pkl")
old_disk = psutil.disk_io_counters()
recent_thrashing_probs=deque(maxlen=5) 
curr_state="NORMAL"
recovers_in=5
recovery_counter=0
safe=0.3
try:
    while True:
        cpu = psutil.cpu_percent(interval=1)
        mem = psutil.virtual_memory()
        swap = psutil.swap_memory()
        new_disk = psutil.disk_io_counters()

        if old_disk and new_disk:
            disk_reads_per_sec = new_disk.read_count - old_disk.read_count
            disk_writes_per_sec = new_disk.write_count - old_disk.write_count
        else:
            disk_reads_per_sec = 0
            disk_writes_per_sec = 0
        old_disk = new_disk
        features = pd.DataFrame([[cpu, mem.percent, swap.percent,
                                   disk_reads_per_sec, disk_writes_per_sec]],
                                 columns=['cpu_percent','ram_percent','swap_percent',
                                          'disk_read_count','disk_write_count'])

        scaled_features = scaler.transform(features)
        probabilities = model.predict_proba(scaled_features)
        print(probabilities)
        p_normal, p_warning, p_thrashing = probabilities[0]
        recent_thrashing_probs.append(p_thrashing)
        avg_thrashing = sum(recent_thrashing_probs) / len(recent_thrashing_probs)
        if curr_state in ("NORMAL", "WARNING"):
            if avg_thrashing >= 0.75:
                curr_state = "THRASHING"
                trigger_working_set_manager(avg_thrashing)
            elif p_warning >= p_normal and p_warning > p_thrashing:
                curr_state = "WARNING"
            else:
                curr_state = "NORMAL"

        elif curr_state == "THRASHING":
            if avg_thrashing < 0.75:
                curr_state = "RECOVERY"
                recovery_counter = 0
        elif curr_state == "RECOVERY":
            if avg_thrashing >= 0.75:
                curr_state = "THRASHING"
                trigger_working_set_manager(avg_thrashing)
                recovery_counter = 0
            elif avg_thrashing < safe:
                recovery_counter += 1
                if recovery_counter >= recovers_in:
                    curr_state = "NORMAL"
                    recovery_counter=0
            else:
                recovery_counter=0

        print(curr_state)
except KeyboardInterrupt:
    print("Running stopped by user")

[[3.43013331e-01 1.00243017e-04 6.56886426e-01]]
NORMAL
[[3.88500539e-01 8.28624134e-05 6.11416598e-01]]
NORMAL
[[2.89126074e-01 1.07540533e-04 7.10766385e-01]]
NORMAL
[[3.60866837e-01 9.22079183e-05 6.39040955e-01]]
NORMAL
[[3.75093264e-01 9.00132301e-05 6.24816723e-01]]
NORMAL
[[3.01942750e-01 1.03444570e-04 6.97953805e-01]]
NORMAL
[[3.44220634e-01 9.94301694e-05 6.55679936e-01]]
NORMAL
[[2.98635352e-01 1.07577588e-04 7.01257070e-01]]
NORMAL
[[3.19435102e-01 9.89219695e-05 6.80465976e-01]]
NORMAL
[[3.36983205e-01 9.55636397e-05 6.62921231e-01]]
NORMAL
[[2.85414098e-01 1.10960524e-04 7.14474941e-01]]
NORMAL
[[2.18887795e-01 1.47223518e-04 7.80964981e-01]]
NORMAL
[[2.76266116e-01 1.13169739e-04 7.23620715e-01]]
NORMAL
[[3.12074353e-01 1.00288913e-04 6.87825358e-01]]
NORMAL
[[2.98632140e-01 1.07102631e-04 7.01260757e-01]]
NORMAL
[[3.30453075e-01 9.62659417e-05 6.69450659e-01]]
NORMAL
[[2.59909064e-01 1.21944538e-04 7.39968992e-01]]
NORMAL
[[2.61103352e-01 1.18539792e-04 7.38778108e-01]]

In [ ]:
model.coef_

array([[-0.75844566, -1.54399309, -1.49108716, -0.44405726, -0.08132077],
       [ 0.25080267,  0.05905057, -2.31692982,  0.55183188,  0.14771064],
       [ 0.50764299,  1.48494252,  3.80801698, -0.10777461, -0.06638987]])